In [2]:
# Install required packages
# !pip install ultralytics opencv-python-headless pillow pyyaml -q

In [3]:
import os
import yaml
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.rcParams['figure.figsize'] = (12, 8)

## 🔍 Device Availability Check

## 1. Dataset Configuration

In [4]:
# Load dataset configuration
data_yaml = 'pcb-defect-dataset/data.yaml'

with open(data_yaml, 'r') as f:
    config = yaml.safe_load(f)

print("Dataset Configuration:")
print(f"Classes: {config['names']}")
print(f"Number of classes: {len(config['names'])}")

Dataset Configuration:
Classes: {0: 'mouse_bite', 1: 'spur', 2: 'missing_hole', 3: 'short', 4: 'open_circuit', 5: 'spurious_copper'}
Number of classes: 6


In [5]:
# Device configuration
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA Version: 12.1


## 2. Train YOLOv8 Model

In [6]:
# Initialize YOLOv8 model
model = YOLO('yolov8n.pt')  # Using nano model for faster training

# Train the model
results = model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    name='pcb_defect_detector',
    patience=10,
    save=True,
    device=device,  # Change to 'cuda' if GPU available
    plots=True,
    verbose=True
)

print("\nTraining completed!")
print(f"Best model saved at: runs/detect/pcb_defect_detector4/weights/best.pt")

engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=pcb-defect-dataset/data.yaml, epochs=50, time=None, patience=10, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=cuda, workers=8, project=None, name=pcb_defect_detector4, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_width=None, format=torchscript, keras=False, optimize=

train: Scanning C:\Users\Akshat\Desktop\Computer-Vision-Engineer-Assignment\task2_quality_inspection\pcb-defect-dataset\train\labels.cache... 6370 images, 2164 backgrounds, 0 corrupt: 100%|██████████| 8534/8534 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Scanning C:\Users\Akshat\Desktop\Computer-Vision-Engineer-Assignment\task2_quality_inspection\pcb-defect-dataset\val\labels.cache... 802 images, 264 backgrounds, 0 corrupt: 100%|██████████| 1066/1066 [00:00<?, ?it/s]


Plotting labels to runs\detect\pcb_defect_detector4\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
TensorBoard: model graph visualization added 
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to runs\detect\pcb_defect_detector4
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

      50/50      2.13G      1.358     0.5762       1.04          7        640: 100%|██████████| 534/534 [00:59<00:00,  9.00it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 34/34 [00:04<00:00,  7.50it/s]

                   all       1066       1595      0.974      0.979      0.988      0.596



50 epochs completed in 0.923 hours.
Optimizer stripped from runs\detect\pcb_defect_detector4\weights\last.pt, 6.3MB
Optimizer stripped from runs\detect\pcb_defect_detector4\weights\best.pt, 6.3MB

Validating runs\detect\pcb_defect_detector4\weights\best.pt...
Ultralytics 8.3.75  Python-3.12.8 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 168 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 34/34 [00:04<00:00,  7.08it/s]


                   all       1066       1595      0.974      0.979      0.988      0.597
            mouse_bite        140        280      0.982      0.983       0.99      0.601
                  spur        130        262      0.977       0.99       0.99      0.591
          missing_hole        118        229      0.981      0.996      0.994      0.644
                 short        158        327      0.969      0.958      0.979      0.601
          open_circuit        135        259      0.979      0.973      0.988      0.554
       spurious_copper        121        238      0.956      0.975       0.99       0.59
Speed: 0.2ms preprocess, 1.6ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to runs\detect\pcb_defect_detector4

Training completed!
Best model saved at: runs/detect/pcb_defect_detector/weights/best.pt


## 3. Model Evaluation

In [9]:
# Load best model
best_model = YOLO('runs/detect/pcb_defect_detector4/weights/best.pt')

# Validate on test set
metrics = best_model.val(data=data_yaml, split='test')

print("\nTest Set Performance:")
print("=" * 50)
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

Ultralytics 8.3.75  Python-3.12.8 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 168 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning C:\Users\Akshat\Desktop\Computer-Vision-Engineer-Assignment\task2_quality_inspection\pcb-defect-dataset\test\labels... 829 images, 239 backgrounds, 0 corrupt: 100%|██████████| 1068/1068 [00:02<00:00, 513.68it/s]


val: New cache created: C:\Users\Akshat\Desktop\Computer-Vision-Engineer-Assignment\task2_quality_inspection\pcb-defect-dataset\test\labels.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 67/67 [00:05<00:00, 11.42it/s]


                   all       1068       1662      0.976      0.988      0.991      0.595
            mouse_bite        131        262      0.967      0.992      0.989      0.591
                  spur        138        279      0.975       0.98      0.991       0.59
          missing_hole        145        283      0.986      0.999      0.994      0.646
                 short        142        275      0.982      0.984      0.991      0.594
          open_circuit        128        265      0.992      0.996      0.995      0.574
       spurious_copper        145        298      0.952       0.98      0.985      0.577
Speed: 0.3ms preprocess, 2.6ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to runs\detect\val

Test Set Performance:
mAP50: 0.9910
mAP50-95: 0.5954
Precision: 0.9756
Recall: 0.9884


## 4. Defect Analysis Script

This section implements the required functionality:
- Analyzes input images
- Detects and localizes defect regions
- Classifies defects with confidence scores
- Outputs (x, y) pixel coordinates of defect centers
- Provides severity assessment

In [16]:
class DefectAnalyzer:
    """
    Automated Quality Inspection System for PCB Defect Detection
    """
    
    def __init__(self, model_path, class_names):
        """
        Initialize the defect analyzer
        
        Args:
            model_path (str): Path to trained YOLO model
            class_names (dict): Dictionary mapping class IDs to names
        """
        self.model = YOLO(model_path)
        self.class_names = class_names
        
        # Severity thresholds based on defect characteristics
        self.severity_thresholds = {
            'area': {'low': 0.01, 'medium': 0.05, 'high': 0.1},
            'confidence': {'low': 0.5, 'medium': 0.7, 'high': 0.9}
        }
        
        # Critical defect types (adjust based on domain knowledge)
        self.critical_defects = ['open_circuit', 'short']
        
    def assess_severity(self, defect_info):
        """
        Assess defect severity based on multiple factors
        
        Args:
            defect_info (dict): Defect information
            
        Returns:
            str: Severity level ('Critical', 'High', 'Medium', 'Low')
        """
        class_name = defect_info['class_name']
        # FIX: use normalized area from dimensions
        area = defect_info['dimensions']['area_normalized']
        confidence = defect_info['confidence']
        
        # Critical defects are always high priority
        if class_name in self.critical_defects:
            return 'Critical'
        
        # Assess based on size and confidence
        severity_score = 0
        
        if area > self.severity_thresholds['area']['high']:
            severity_score += 3
        elif area > self.severity_thresholds['area']['medium']:
            severity_score += 2
        elif area > self.severity_thresholds['area']['low']:
            severity_score += 1
        
        if confidence > self.severity_thresholds['confidence']['high']:
            severity_score += 2
        elif confidence > self.severity_thresholds['confidence']['medium']:
            severity_score += 1
        
        # Map score to severity
        if severity_score >= 4:
            return 'High'
        elif severity_score >= 2:
            return 'Medium'
        else:
            return 'Low'
    
    def analyze_image(self, image_path, conf_threshold=0.25):
        """
        Analyze a single image for defects
        
        Args:
            image_path (str): Path to input image
            conf_threshold (float): Confidence threshold for detections
            
        Returns:
            dict: Analysis results including defects and overall assessment
        """
        # Load image
        img = cv2.imread(str(image_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # Run inference
        results = self.model(img, conf=conf_threshold, verbose=False)[0]
        
        # Parse detections
        defects = []
        for box in results.boxes:
            # Extract box coordinates
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            
            # Calculate center coordinates
            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)
            
            # Calculate normalized area
            box_width = (x2 - x1) / w
            box_height = (y2 - y1) / h
            area = box_width * box_height
            
            # Get class and confidence
            class_id = int(box.cls[0].cpu().numpy())
            confidence = float(box.conf[0].cpu().numpy())
            class_name = self.class_names[class_id]
            
            # Create defect info
            defect_info = {
                'class_id': class_id,
                'class_name': class_name,
                'confidence': confidence,
                'bbox': {
                    'x1': int(x1), 'y1': int(y1),
                    'x2': int(x2), 'y2': int(y2)
                },
                'center': {'x': center_x, 'y': center_y},
                'dimensions': {
                    'width_px': int(x2 - x1),
                    'height_px': int(y2 - y1),
                    'area_normalized': float(area)
                }
            }
            
            # Assess severity
            defect_info['severity'] = self.assess_severity(defect_info)
            
            defects.append(defect_info)
        
        # Overall assessment
        assessment = {
            'image_path': str(image_path),
            'image_dimensions': {'width': w, 'height': h},
            'timestamp': datetime.now().isoformat(),
            'total_defects': len(defects),
            'defects': defects,
            'quality_status': 'PASS' if len(defects) == 0 else 'FAIL'
        }
        
        # Add severity breakdown
        severity_counts = {'Critical': 0, 'High': 0, 'Medium': 0, 'Low': 0}
        for defect in defects:
            severity_counts[defect['severity']] += 1
        assessment['severity_breakdown'] = severity_counts
        
        return assessment
    
    def visualize_results(self, image_path, analysis_results, save_path=None):
        """
        Visualize detection results with annotations
        
        Args:
            image_path (str): Path to original image
            analysis_results (dict): Analysis results from analyze_image()
            save_path (str, optional): Path to save annotated image
            
        Returns:
            numpy.ndarray: Annotated image
        """
        # Load image
        img = cv2.imread(str(image_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Color map for severity
        severity_colors = {
            'Critical': (255, 0, 0),      # Red
            'High': (255, 128, 0),        # Orange
            'Medium': (255, 255, 0),      # Yellow
            'Low': (0, 255, 0)            # Green
        }
        
        # Draw detections
        for defect in analysis_results['defects']:
            bbox = defect['bbox']
            center = defect['center']
            severity = defect['severity']
            color = severity_colors.get(severity, (128, 128, 128))
            
            # Draw bounding box
            cv2.rectangle(img, (bbox['x1'], bbox['y1']), 
                         (bbox['x2'], bbox['y2']), color, 2)
            
            # Draw center point
            cv2.circle(img, (center['x'], center['y']), 5, color, -1)
            
            # Add label
            label = f"{defect['class_name']}"
            label += f" {defect['confidence']:.2f}"
            label += f" [{severity}]"
            
            # Background for text
            (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            cv2.rectangle(img, (bbox['x1'], bbox['y1'] - text_h - 10),
                         (bbox['x1'] + text_w, bbox['y1']), color, -1)
            
            cv2.putText(img, label, (bbox['x1'], bbox['y1'] - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        
        # Add overall status
        status_color = (0, 255, 0) if analysis_results['quality_status'] == 'PASS' else (255, 0, 0)
        status_text = f"Status: {analysis_results['quality_status']} | Defects: {analysis_results['total_defects']}"
        cv2.putText(img, status_text, (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, status_color, 2)
        
        # Save if requested
        if save_path:
            img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            cv2.imwrite(save_path, img_bgr)
        
        return img
    
    def generate_report(self, analysis_results, output_path=None):
        """
        Generate detailed inspection report
        
        Args:
            analysis_results (dict): Analysis results
            output_path (str, optional): Path to save JSON report
            
        Returns:
            str: Formatted report string
        """
        report = []
        report.append("=" * 70)
        report.append("PCB DEFECT INSPECTION REPORT")
        report.append("=" * 70)
        report.append(f"\nImage: {analysis_results['image_path']}")
        report.append(f"Timestamp: {analysis_results['timestamp']}")
        report.append(f"Quality Status: {analysis_results['quality_status']}")
        report.append(f"\nTotal Defects Detected: {analysis_results['total_defects']}")
        
        # Severity breakdown
        report.append("\nSeverity Breakdown:")
        for severity, count in analysis_results['severity_breakdown'].items():
            if count > 0:
                report.append(f"  {severity}: {count}")
        
        # Detailed defect information
        if analysis_results['defects']:
            report.append("\nDetailed Defect Information:")
            report.append("-" * 70)
            
            for i, defect in enumerate(analysis_results['defects'], 1):
                report.append(f"\nDefect #{i}:")
                report.append(f"  Type: {defect['class_name']}")
                report.append(f"  Confidence: {defect['confidence']:.3f}")
                report.append(f"  Center (x, y): ({defect['center']['x']}, {defect['center']['y']})")
                report.append(f"  Size: {defect['dimensions']['width_px']}×{defect['dimensions']['height_px']} px")
                report.append(f"  Normalized Area: {defect['dimensions']['area_normalized']:.4f}")
                report.append(f"  Severity: {defect['severity']}")
        
        report.append("\n" + "=" * 70)
        
        report_text = "\n".join(report)
        
        # Save JSON report if requested
        if output_path:
            with open(output_path, 'w') as f:
                json.dump(analysis_results, f, indent=2)
        
        return report_text

# Initialize analyzer
analyzer = DefectAnalyzer(
    model_path='runs/detect/pcb_defect_detector4/weights/best.pt',
    class_names=config['names']
)

print("Defect Analyzer initialized successfully!")

Defect Analyzer initialized successfully!


## 5. Test the Inspection System

In [17]:
# Test on sample images from test set
test_images_dir = Path('pcb-defect-dataset/test/images')
sample_images = list(test_images_dir.glob('*.jpg'))[:6]  # Get 6 samples

# Create output directory
output_dir = Path('inspection_results')
output_dir.mkdir(exist_ok=True)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

print("Running defect inspection on sample images...\n")

for idx, img_path in enumerate(sample_images):
    # Analyze image
    results = analyzer.analyze_image(img_path, conf_threshold=0.3)
    
    # Generate report
    report = analyzer.generate_report(results)
    print(f"\n{report}")
    
    # Save JSON report
    json_path = output_dir / f"{img_path.stem}_report.json"
    with open(json_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    # Visualize
    annotated_img = analyzer.visualize_results(
        img_path, 
        results,
        save_path=output_dir / f"{img_path.stem}_annotated.jpg"
    )
    
    # Display
    axes[idx].imshow(annotated_img)
    axes[idx].set_title(f"{img_path.name}\n{results['quality_status']} - {results['total_defects']} defects", 
                       fontsize=10)
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(output_dir / 'inspection_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Results saved to: {output_dir.absolute()}")

Running defect inspection on sample images...


PCB DEFECT INSPECTION REPORT

Image: pcb-defect-dataset\test\images\light_01_missing_hole_02_3_600.jpg
Timestamp: 2026-01-10T02:59:07.777556
Quality Status: PASS

Total Defects Detected: 0

Severity Breakdown:


PCB DEFECT INSPECTION REPORT

Image: pcb-defect-dataset\test\images\light_01_missing_hole_05_2_600.jpg
Timestamp: 2026-01-10T02:59:07.863574
Quality Status: PASS

Total Defects Detected: 0

Severity Breakdown:


PCB DEFECT INSPECTION REPORT

Image: pcb-defect-dataset\test\images\light_01_missing_hole_06_3_600.jpg
Timestamp: 2026-01-10T02:59:07.926939
Quality Status: PASS

Total Defects Detected: 0

Severity Breakdown:


PCB DEFECT INSPECTION REPORT

Image: pcb-defect-dataset\test\images\light_01_missing_hole_07_1_600.jpg
Timestamp: 2026-01-10T02:59:07.950197
Quality Status: PASS

Total Defects Detected: 0

Severity Breakdown:


PCB DEFECT INSPECTION REPORT

Image: pcb-defect-dataset\test\images\light_01_missing_hole_08_2_600.jpg
T

<Figure size 1800x1200 with 6 Axes>


✓ Results saved to: c:\Users\Akshat\Desktop\Computer-Vision-Engineer-Assignment\task2_quality_inspection\inspection_results


## 6. Batch Processing Script

Process multiple images and generate comprehensive reports

In [18]:
def batch_inspect(input_dir, output_dir, conf_threshold=0.3):
    """
    Perform batch inspection on all images in a directory
    
    Args:
        input_dir (str/Path): Directory containing images to inspect
        output_dir (str/Path): Directory to save results
        conf_threshold (float): Confidence threshold
        
    Returns:
        dict: Summary statistics
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    # Create subdirectories
    (output_dir / 'annotated').mkdir(exist_ok=True)
    (output_dir / 'reports').mkdir(exist_ok=True)
    
    image_files = list(input_dir.glob('*.jpg'))
    
    all_results = []
    total_defects = 0
    defect_types_count = {name: 0 for name in config['names'].values()}
    
    print(f"Processing {len(image_files)} images...")
    
    for img_path in image_files:
        # Analyze
        results = analyzer.analyze_image(img_path, conf_threshold)
        all_results.append(results)
        
        # Count defects
        total_defects += results['total_defects']
        for defect in results['defects']:
            defect_types_count[defect['class_name']] += 1
        
        # Save annotated image
        analyzer.visualize_results(
            img_path,
            results,
            save_path=output_dir / 'annotated' / f"{img_path.stem}_annotated.jpg"
        )
        
        # Save JSON report
        with open(output_dir / 'reports' / f"{img_path.stem}_report.json", 'w') as f:
            json.dump(results, f, indent=2)
    
    # Generate summary
    summary = {
        'total_images_processed': len(image_files),
        'total_defects_found': total_defects,
        'images_with_defects': sum(1 for r in all_results if r['total_defects'] > 0),
        'pass_rate': sum(1 for r in all_results if r['quality_status'] == 'PASS') / len(image_files),
        'defect_type_distribution': defect_types_count,
        'average_defects_per_image': total_defects / len(image_files)
    }
    
    # Save summary
    with open(output_dir / 'batch_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    
    # Print summary
    print("\n" + "=" * 70)
    print("BATCH INSPECTION SUMMARY")
    print("=" * 70)
    print(f"Total Images Processed: {summary['total_images_processed']}")
    print(f"Total Defects Found: {summary['total_defects_found']}")
    print(f"Images with Defects: {summary['images_with_defects']}")
    print(f"Pass Rate: {summary['pass_rate']:.2%}")
    print(f"Average Defects per Image: {summary['average_defects_per_image']:.2f}")
    print("\nDefect Type Distribution:")
    for defect_type, count in defect_types_count.items():
        if count > 0:
            print(f"  {defect_type}: {count}")
    print("=" * 70)
    
    return summary

# Example: Process test set
test_summary = batch_inspect(
    input_dir='pcb-defect-dataset/test/images',
    output_dir='batch_inspection_results',
    conf_threshold=0.3
)

Processing 1068 images...

BATCH INSPECTION SUMMARY
Total Images Processed: 1068
Total Defects Found: 1685
Images with Defects: 825
Pass Rate: 22.75%
Average Defects per Image: 1.58

Defect Type Distribution:
  mouse_bite: 269
  spur: 279
  missing_hole: 289
  short: 276
  open_circuit: 265
  spurious_copper: 307


## 7. Export Standalone Inspection Script

Create a standalone Python script for production use

In [ ]:
standalone_script = '''#!/usr/bin/env python3
"""
PCB Defect Detection - Automated Quality Inspection System

This script performs automated defect detection on PCB images using YOLOv8.

Requirements:
    - Analyzes input images
    - Detects and localizes defect regions  
    - Classifies defects with confidence scores
    - Outputs (x, y) pixel coordinates of defect centers
    - Provides severity assessment

Usage:
    python inspect_pcb.py --image path/to/image.jpg
    python inspect_pcb.py --batch path/to/images/ --output results/
"""

import argparse
import os
import json
import cv2
import numpy as np
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO


class DefectAnalyzer:
    """Automated Quality Inspection System for PCB Defect Detection"""
    
    def __init__(self, model_path, class_names):
        self.model = YOLO(model_path)
        self.class_names = class_names
        self.critical_defects = ['open_circuit', 'short']
        
    def assess_severity(self, defect_info):
        """Assess defect severity"""
        class_name = defect_info['class_name']
        area = defect_info['dimensions']['area_normalized']
        confidence = defect_info['confidence']
        
        if class_name in self.critical_defects:
            return 'Critical'
        
        severity_score = 0
        if area > 0.1: severity_score += 3
        elif area > 0.05: severity_score += 2
        elif area > 0.01: severity_score += 1
        
        if confidence > 0.9: severity_score += 2
        elif confidence > 0.7: severity_score += 1
        
        return 'High' if severity_score >= 4 else 'Medium' if severity_score >= 2 else 'Low'
    
    def analyze_image(self, image_path, conf_threshold=0.25):
        """Analyze a single image for defects"""
        img = cv2.imread(str(image_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        results = self.model(img, conf=conf_threshold, verbose=False)[0]
        
        defects = []
        for box in results.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)
            
            box_width = (x2 - x1) / w
            box_height = (y2 - y1) / h
            area = box_width * box_height
            
            class_id = int(box.cls[0].cpu().numpy())
            confidence = float(box.conf[0].cpu().numpy())
            
            defect_info = {
                'class_id': class_id,
                'class_name': self.class_names[class_id],
                'confidence': confidence,
                'bbox': {'x1': int(x1), 'y1': int(y1), 'x2': int(x2), 'y2': int(y2)},
                'center': {'x': center_x, 'y': center_y},
                'dimensions': {
                    'width_px': int(x2 - x1),
                    'height_px': int(y2 - y1),
                    'area_normalized': float(area)
                }
            }
            
            defect_info['severity'] = self.assess_severity(defect_info)
            defects.append(defect_info)
        
        severity_counts = {'Critical': 0, 'High': 0, 'Medium': 0, 'Low': 0}
        for defect in defects:
            severity_counts[defect['severity']] += 1
        
        return {
            'image_path': str(image_path),
            'image_dimensions': {'width': w, 'height': h},
            'timestamp': datetime.now().isoformat(),
            'total_defects': len(defects),
            'defects': defects,
            'quality_status': 'PASS' if len(defects) == 0 else 'FAIL',
            'severity_breakdown': severity_counts
        }
    
    def visualize_results(self, image_path, analysis_results, save_path=None):
        """Visualize detection results"""
        img = cv2.imread(str(image_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        severity_colors = {
            'Critical': (255, 0, 0), 'High': (255, 128, 0),
            'Medium': (255, 255, 0), 'Low': (0, 255, 0)
        }
        
        for defect in analysis_results['defects']:
            bbox = defect['bbox']
            center = defect['center']
            color = severity_colors.get(defect['severity'], (128, 128, 128))
            
            cv2.rectangle(img, (bbox['x1'], bbox['y1']), (bbox['x2'], bbox['y2']), color, 2)
            cv2.circle(img, (center['x'], center['y']), 5, color, -1)
            
            label = f"{defect['class_name']} {defect['confidence']:.2f} [{defect['severity']}]"
            cv2.putText(img, label, (bbox['x1'], bbox['y1'] - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        
        status_color = (0, 255, 0) if analysis_results['quality_status'] == 'PASS' else (255, 0, 0)
        status_text = f"Status: {analysis_results['quality_status']} | Defects: {analysis_results['total_defects']}"
        cv2.putText(img, status_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, status_color, 2)
        
        if save_path:
            img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            cv2.imwrite(save_path, img_bgr)
        
        return img


def main():
    parser = argparse.ArgumentParser(description='PCB Defect Detection System')
    parser.add_argument('--image', type=str, help='Path to single image')
    parser.add_argument('--batch', type=str, help='Path to directory of images')
    parser.add_argument('--output', type=str, default='inspection_results', help='Output directory')
    parser.add_argument('--model', type=str, default='runs/detect/pcb_defect_detector_fast/weights/best.pt', 
                       help='Path to model weights')
    parser.add_argument('--conf', type=float, default=0.3, help='Confidence threshold')
    args = parser.parse_args()
    
    # Class names
    class_names = {
        0: 'mouse_bite', 1: 'spur', 2: 'missing_hole',
        3: 'short', 4: 'open_circuit', 5: 'spurious_copper'
    }
    
    # Initialize analyzer
    analyzer = DefectAnalyzer(args.model, class_names)
    output_dir = Path(args.output)
    output_dir.mkdir(exist_ok=True)
    
    if args.image:
        # Single image processing
        results = analyzer.analyze_image(args.image, args.conf)
        
        print("\\n" + "="*70)
        print(f"Image: {results['image_path']}")
        print(f"Status: {results['quality_status']}")
        print(f"Defects: {results['total_defects']}")
        
        for i, defect in enumerate(results['defects'], 1):
            print(f"\\nDefect #{i}:")
            print(f"  Type: {defect['class_name']}")
            print(f"  Center: ({defect['center']['x']}, {defect['center']['y']})")
            print(f"  Confidence: {defect['confidence']:.3f}")
            print(f"  Severity: {defect['severity']}")
        
        analyzer.visualize_results(args.image, results, 
                                  save_path=output_dir / 'annotated.jpg')
        
        with open(output_dir / 'report.json', 'w') as f:
            json.dump(results, f, indent=2)
        
        print(f"\\nResults saved to: {output_dir.absolute()}")
    
    elif args.batch:
        # Batch processing
        batch_dir = Path(args.batch)
        images = list(batch_dir.glob('*.jpg'))
        
        print(f"Processing {len(images)} images...")
        
        for img_path in images:
            results = analyzer.analyze_image(img_path, args.conf)
            
            analyzer.visualize_results(img_path, results,
                                      save_path=output_dir / f"{img_path.stem}_annotated.jpg")
            
            with open(output_dir / f"{img_path.stem}_report.json", 'w') as f:
                json.dump(results, f, indent=2)
        
        print(f"\\n✓ Processed {len(images)} images. Results saved to: {output_dir.absolute()}")
    
    else:
        parser.print_help()


if __name__ == '__main__':
    main()
'''

# Save the script
with open('inspect_pcb.py', 'w') as f:
    f.write(standalone_script)

print("✓ Standalone script saved as: inspect_pcb.py")
print("\nUsage examples:")
print("  Single image: python inspect_pcb.py --image sample.jpg")
print("  Batch mode:   python inspect_pcb.py --batch images_folder/ --output results/")